<a href="https://colab.research.google.com/github/hapantherim5-sudo/HW3_Panthers/blob/main/HW3_Panthers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install project dependencies
!pip install -q nltk scikit-learn sentence-transformers gradio
!pip install -q paho-mqtt opencv-python-headless plotly ipywidgets
!pip install -q beautifulsoup4 pypdf google-genai firebase-admin


In [ ]:
import io, json, re, random, time, threading, os, requests
from datetime import datetime, timedelta
from collections import Counter
import json as json_lib
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, clear_output, display
from PIL import Image
import ipywidgets as widgets
from bs4 import BeautifulSoup
from pypdf import PdfReader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)
nltk.download('punkt_tab', quiet=True)

try:
    from sentence_transformers import SentenceTransformer
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False

try:
    import gradio as gr
    GRADIO_AVAILABLE = True
except ImportError:
    GRADIO_AVAILABLE = False

try:
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

try:
    import paho.mqtt.client as mqtt
    MQTT_AVAILABLE = True
except ImportError:
    MQTT_AVAILABLE = False

print("✅ All libraries loaded!")


In [ ]:
# Secure Gemini connection using Google Colab Secrets
# In Colab: Secrets (key icon) -> add GEMINI_API_KEY -> enable Notebook access

from google.colab import userdata
from google import genai
from google.genai import types, errors

GEMINI_MODEL = "gemini-3.1-flash-lite"

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI_API_KEY was not found in Colab Secrets.")

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    print("✅ Gemini connected securely")

except Exception as e:
    gemini_client = None
    print(f"⚠️ Gemini is not connected: {e}")


def generate_with_retry(client, prompt, max_retries=4):
    """Calls Gemini with short exponential backoff for temporary 503/429 errors."""
    if client is None:
        raise RuntimeError("Gemini client is not connected.")

    last_error = None

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.2,
                    max_output_tokens=350
                )
            )

            text = getattr(response, "text", None)
            if text:
                return text.strip()

            raise RuntimeError("Gemini returned an empty response.")

        except (errors.ServerError, errors.ClientError) as e:
            last_error = e
            status_code = getattr(e, "code", None)

            # Retry only temporary overload/rate-limit errors.
            if status_code not in (429, 503) or attempt == max_retries - 1:
                raise

            wait_seconds = 2 ** attempt
            print(f"⏳ Gemini is temporarily busy. Retrying in {wait_seconds}s...")
            time.sleep(wait_seconds)

    raise last_error


In [ ]:
import json
import firebase_admin

from firebase_admin import credentials, firestore
from google.colab import userdata


def connect_to_firebase():
    try:
        firebase_secret = userdata.get("FIREBASE_SERVICE_ACCOUNT")

        if not firebase_secret:
            raise ValueError(
                "FIREBASE_SERVICE_ACCOUNT was not found in Colab Secrets."
            )

        service_account_info = json.loads(firebase_secret)

        # מונע אתחול כפול אם מריצים את התא שוב
        if not firebase_admin._apps:
            credential = credentials.Certificate(service_account_info)
            firebase_admin.initialize_app(credential)

        db = firestore.client()

        print("✅ Firebase Firestore connected successfully")
        return db

    except json.JSONDecodeError:
        print("❌ The Firebase secret is not valid JSON.")
        return None

    except Exception as error:
        print(f"❌ Firebase connection failed: {error}")
        return None


firebase_db = connect_to_firebase()

In [ ]:
# Firebase connection status
if firebase_db is None:
    print("❌ Firebase is not connected.")
else:
    print("✅ Firebase is ready. Collections will be created automatically when data is saved.")


In [ ]:
# Define color theme, research documents, and stop words

# Calendula-inspired color palette (yellow / orange / brown)
COLORS = {
    "primary":   "#E8860A",
    "secondary": "#F5C518",
    "accent":    "#C0392B",
    "brown":     "#7B3F00",
    "light":     "#FFF3CD",
    "healthy":   "#F39C12",
    "dry":       "#C0392B",
    "sick":      "#7B3F00",
    "warning":   "#E67E22",
    "bg":        "#FDF6E3",
    "text":      "#3E1C00",
}

# Five academic papers about Calendula officinalis used as the knowledge base
documents = {
    "Doc1": {
        "Title":    "An Updated Review on the Multifaceted Therapeutic Potential of Calendula officinalis L.",
        "URL":      "https://www.mdpi.com/1424-8247/16/4/611",
        "authors":  "Lima, R.R. et al.",
        "journal":  "Pharmaceuticals",
        "year":     2023,
        "topic":    "Therapeutic properties, anti-inflammatory, antioxidant, wound healing",
        "content":  "calendula officinalis flavonoids carotenoids triterpenoids saponins anti-inflammatory antioxidant wound healing antimicrobial medicinal plant extract phytochemical pot marigold therapeutic properties flowers",
        "abstract": "Calendula officinalis contains flavonoids triterpenoids glycosides saponins carotenoids volatile oil amino acids steroids. Anti-inflammatory anti-cancer antihelminthic antidiabetes wound healing hepatoprotective antioxidant activities. Used for burns gastrointestinal gynecological ocular skin conditions. Molecular mechanisms clinical studies traditional medicine therapeutic potential."
    },
    "Doc2": {
        "Title":    "Calendula officinalis: A New Natural Host of Pseudomonas viridiflava in Italy",
        "URL":      "https://apsjournals.apsnet.org/doi/abs/10.1094/PDIS-08-11-0691",
        "authors":  "Moretti, C., Fakhr, R., Buonaurio, R.",
        "journal":  "Plant Disease",
        "year":     2012,
        "topic":    "Plant disease, bacterial pathogen, Pseudomonas",
        "content":  "calendula officinalis pseudomonas viridiflava bacterial pathogen necrotic spots diseases Italy plant infection leaves symptoms isolated strains",
        "abstract": "Calendula officinalis infected by Pseudomonas viridiflava bacterial pathogen. Necrotic spots chlorotic halos observed on leaves. Bacterial colonies isolated from diseased leaf tissues. Gram negative fluorescent strains pathogenicity test confirmed. First report plant disease Italy infection symptoms."
    },
    "Doc3": {
        "Title":    "Activity of Essential Oils and Plant Extracts as Biofungicides for Suppression of Soil-Borne Fungi Associated with Root Rot and Wilt of Marigold",
        "URL":      "https://www.mdpi.com/2311-7524/9/2/222",
        "authors":  "Battaglia, M.L.",
        "journal":  "Horticulturae",
        "year":     2023,
        "topic":    "Plant disease control, biofungicides, root rot, wilt",
        "content":  "calendula officinalis essential oils plant extracts biofungicide antifungal antioxidant antimicrobial root rot wilt fusarium pathogens phytochemical marigold soil treatments inhibition growth",
        "abstract": "Essential oils plant extracts biofungicides suppress soil-borne fungi root rot wilt marigold calendula. Rhizoctonia solani Sclerotinia sclerotiorum Fusarium oxysporum pathogens tested. Cinnamon peppermint clove thyme oils inhibition mycelium growth. Eco-friendly alternatives synthetic fungicides sustainable control antifungal antimicrobial."
    },
    "Doc4": {
        "Title":    "Assessment of Plant Development, Morphology and Flavonoid Content in Different Cultivation Treatments of Calendula officinalis L.",
        "URL":      "https://www.scielo.br/j/rbfar/a/7dJBLkS3tcH8V9WTpDYz9dk/?lang=en",
        "authors":  "SciELO Brazil Authors",
        "journal":  "Revista Brasileira de Farmacognosia",
        "year":     2010,
        "topic":    "Morphology, cultivation, flavonoids, plant development",
        "content":  "calendula officinalis morphology flavonoids cultivation fertilisation plant development phytochemical pot marigold medicinal flowers tubular treatments organic",
        "abstract": "Calendula officinalis morphology flavonoid content cultivation treatments liming organic fertilizer NPK chemical fertilizer. Plant development flowers biomass fresh mass homogeneity. Morphoanatomical variations tubular flowers yellow brown centres. Phytochemical parameters cultivation conditions growth."
    },
    "Doc5": {
        "Title":    "Pot Marigold (Calendula officinalis) Medicinal Usage and Cultivation",
        "URL":      "https://academicjournals.org/journal/SRE/article-full-text-pdf/49E0E0929070.pdf",
        "authors":  "Mohammad, S.M., Kashani, H.H.",
        "journal":  "Scientific Research and Essays",
        "year":     2012,
        "topic":    "Medicinal usage, cultivation, history, pharmacology",
        "content":  "calendula officinalis medicinal pot marigold cultivation flavonoids carotenoids triterpenoids saponins essential oils anti-inflammatory antioxidant wound healing plant extract properties flowers historical",
        "abstract": "Calendula officinalis pot marigold medicinal usage cultivation history. Flavonoids carotenoids triterpenoids saponins essential oils chemical composition. Anti-inflammatory antioxidant wound healing antimicrobial antiviral properties. Traditional medicine skin diseases gastro-intestinal gynecological treatments. Pharmacological studies allergic side effects."
    }
}

# Common English words excluded from the index to reduce noise
stop_words = {
    "a","an","the","and","or","but","of","in","on","at","to","for","with",
    "by","from","as","is","are","was","were","be","been","it","its","this",
    "that","these","those","we","our","their","they","which","who","has",
    "have","had","not","also","using","used","use","may","can","all",
    "study","studies","results","show","showed","effect","effects","based",
    "significant","however","although","therefore","thus"
}

print("✅ Constants defined!")

In [ ]:
def clean_scraped_text(text):
    """Cleans text extracted from websites/PDFs."""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\-\.\,\;\:\(\)]', ' ', text)
    return text.strip()


def extract_text_from_html(url):
    """Downloads an HTML page and extracts visible text."""
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove non-content elements
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(separator=" ")
    return clean_scraped_text(text)


def extract_text_from_pdf(url):
    """Downloads a PDF and extracts text from all pages."""
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers, timeout=60)
    response.raise_for_status()

    pdf_file = io.BytesIO(response.content)
    reader = PdfReader(pdf_file)

    pages_text = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            pages_text.append(page_text)

    return clean_scraped_text(" ".join(pages_text))


def extract_text_from_url(url):
    """Extracts text from either an HTML page or a PDF URL."""
    try:
        if url.lower().endswith(".pdf"):
            return extract_text_from_pdf(url)

        return extract_text_from_html(url)

    except Exception as e:
        print(f"⚠️ Could not scrape {url}: {e}")
        return ""


def enrich_documents_from_web(docs):
    """Goes into each paper URL, extracts real website/PDF text, and updates content."""
    enriched_docs = {}

    for doc_id, doc in docs.items():
        url = doc.get("URL", "")
        print(f"🌐 Scraping {doc_id}: {url}")

        scraped_text = extract_text_from_url(url)

        if len(scraped_text) > 300:
            doc["content"] = scraped_text
            doc["abstract"] = scraped_text[:4000]
            doc["scraped_from_web"] = True
            doc["word_count"] = len(scraped_text.split())
            print(f"✅ {doc_id}: scraped {doc['word_count']} words")
        else:
            doc["scraped_from_web"] = False
            doc["word_count"] = len(doc.get("content", "").split())
            print(f"⚠️ {doc_id}: using fallback manual content")

        enriched_docs[doc_id] = doc

    return enriched_docs

In [ ]:
# Inverted index classes — IndexService, QueryService, ResultService

class IndexService:
    """Builds and queries an inverted index with lemmatization support."""

    def __init__(self):
        self.documents  = {}
        self.index      = {}
        self.lemmatizer = WordNetLemmatizer()

    def lemmatize_word(self, word):
        """Returns the base (lemma) form of a word."""
        return self.lemmatizer.lemmatize(word.lower())

    def add_document(self, doc_id, doc_data):
        """Tokenizes and indexes a document after filtering stop words and lemmatizing."""
        self.documents[doc_id] = doc_data
        tokens = word_tokenize(doc_data['content'].lower())
        for word in tokens:
            if word.isalpha() and word not in stop_words:
                lemma = self.lemmatize_word(word)
                if lemma not in self.index:
                    self.index[lemma] = set()
                self.index[lemma].add(doc_id)

    def get_document(self, doc_id):
        """Retrieves a document by its ID."""
        return self.documents.get(doc_id)

    def search_word(self, word):
        """Returns all document IDs that contain the given word (after lemmatization)."""
        return list(self.index.get(self.lemmatize_word(word), set()))

    def print_index(self):
        """Prints the full inverted index as a formatted table."""
        print("=" * 55)
        print(f"{'TERM (lemma)':<25} {'DocIDs'}")
        print("=" * 55)
        for term, doc_ids in sorted(self.index.items()):
            print(f"{term:<25} {sorted(doc_ids)}")
        print("=" * 55)

    def to_dataframe(self):
        """Returns the index as a DataFrame with columns: term, DocIDs."""
        import pandas as pd
        return pd.DataFrame([
            {"term": term, "DocIDs": sorted(doc_ids)}
            for term, doc_ids in sorted(self.index.items())
        ])

class QueryService:
    """Handles multi-term queries using intersection over the inverted index."""

    def __init__(self, index_service):
        self.index_service = index_service
        self.queries = {}

    def create_query(self, terms):
        """Searches for documents matching ALL given terms (AND logic)."""
        query_id = str(len(self.queries) + 1)

        clean_terms = [
            self.index_service.lemmatize_word(term.lower().strip())
            for term in terms
            if term and term.strip()
        ]

        if not clean_terms:
            results = set()
        else:
            results = set(self.index_service.search_word(clean_terms[0]))

            for term in clean_terms[1:]:
                results &= set(self.index_service.search_word(term))

        query = {
            'id': query_id,
            'terms': clean_terms,
            'results': sorted(list(results))
        }

        self.queries[query_id] = query
        return query



class ResultService:
    """Formats and displays query results."""

    def __init__(self, index_service, query_service):
        self.index_service = index_service
        self.query_service = query_service

    def format_results(self, query_id):
        """Prints the title and URL of each document matched by the given query."""
        query = self.query_service.queries.get(query_id)
        if not query:
            return print("Query not found.")
        print(f"\nResults for: {query['terms']}")
        print("-" * 55)
        if not query['results']:
            print("No documents found.")
            return
        for doc_id in sorted(query['results']):
            doc = self.index_service.get_document(doc_id)
            if doc:
                print(f"  {doc_id}: {doc['Title']}")
                print(f"  URL: {doc['URL']}")
                print()

print("✅ Index classes defined!")

In [ ]:
# Firebase persistence layer
# Centralizes all Firestore reads/writes so the notebook does not rebuild data on every run.

from datetime import datetime, timezone
import hashlib


def _to_plain_value(value):
    """Converts pandas/numpy/datetime values to Firestore-safe Python values."""
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    if isinstance(value, datetime):
        return value
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, set):
        return sorted(value)
    if isinstance(value, dict):
        return {str(k): _to_plain_value(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_to_plain_value(v) for v in value]
    if pd.isna(value):
        return None
    return value


class FirebaseRepository:
    """Repository for all persistent application data."""

    def __init__(self, db):
        if db is None:
            raise ValueError("Firestore client is required.")
        self.db = db

    # ---------- Research papers ----------
    def save_documents(self, documents):
        batch = self.db.batch()
        collection = self.db.collection("research_documents")

        for doc_id, document in documents.items():
            ref = collection.document(str(doc_id))
            payload = _to_plain_value(document)

            # Keep every Firestore document comfortably below the 1 MiB limit.
            payload["content"] = str(payload.get("content", ""))[:250_000]
            payload["abstract"] = str(payload.get("abstract", ""))[:8_000]
            payload["doc_id"] = str(doc_id)
            payload["updated_at"] = firestore.SERVER_TIMESTAMP
            batch.set(ref, payload)

        batch.commit()

        self.db.collection("system_metadata").document("research_documents").set({
            "count": len(documents),
            "updated_at": firestore.SERVER_TIMESTAMP
        })

    def load_documents(self):
        snapshots = self.db.collection("research_documents").stream()
        documents = {}

        for snapshot in snapshots:
            data = snapshot.to_dict()
            data.pop("updated_at", None)
            data.pop("doc_id", None)
            documents[snapshot.id] = data

        return documents

    # ---------- Inverted index ----------
    def save_inverted_index(self, index_service, chunk_size=350):
        index_data = {
            term: sorted(list(doc_ids))
            for term, doc_ids in index_service.index.items()
        }

        # Delete old chunks before writing the current version.
        old_chunks = self.db.collection("inverted_index_chunks").stream()
        delete_batch = self.db.batch()
        delete_count = 0
        for snapshot in old_chunks:
            delete_batch.delete(snapshot.reference)
            delete_count += 1
            if delete_count % 400 == 0:
                delete_batch.commit()
                delete_batch = self.db.batch()
        delete_batch.commit()

        terms = sorted(index_data)
        batch = self.db.batch()
        chunk_count = 0

        for start in range(0, len(terms), chunk_size):
            chunk_terms = terms[start:start + chunk_size]
            chunk_payload = {term: index_data[term] for term in chunk_terms}
            ref = self.db.collection("inverted_index_chunks").document(
                f"chunk_{chunk_count:04d}"
            )
            batch.set(ref, {
                "terms": chunk_payload,
                "term_count": len(chunk_payload),
                "updated_at": firestore.SERVER_TIMESTAMP
            })
            chunk_count += 1

        batch.commit()

        self.db.collection("system_metadata").document("inverted_index").set({
            "terms_count": len(index_data),
            "documents_count": len(index_service.documents),
            "chunk_count": chunk_count,
            "updated_at": firestore.SERVER_TIMESTAMP
        })

    def load_inverted_index(self, index_service, documents):
        snapshots = self.db.collection("inverted_index_chunks").stream()

        loaded_index = {}
        for snapshot in snapshots:
            chunk = snapshot.to_dict().get("terms", {})
            for term, doc_ids in chunk.items():
                loaded_index[term] = set(doc_ids)

        if not loaded_index:
            return False

        index_service.index = loaded_index
        index_service.documents = documents
        return True

    # ---------- RAG vector cache ----------
    def save_rag_cache(self, rag_system):
        batch = self.db.batch()
        collection = self.db.collection("rag_vector_cache")

        # Remove previous cache documents.
        for snapshot in collection.stream():
            batch.delete(snapshot.reference)
        batch.commit()

        batch = self.db.batch()
        for i, doc_id in enumerate(rag_system.collection.ids):
            ref = collection.document(str(doc_id))
            batch.set(ref, {
                "doc_id": str(doc_id),
                "searchable_text": rag_system.collection.documents[i],
                "embedding": _to_plain_value(rag_system.collection.embeddings[i]),
                "metadata": _to_plain_value(rag_system.collection.metadatas[i]),
                "updated_at": firestore.SERVER_TIMESTAMP
            })
        batch.commit()

        self.db.collection("system_metadata").document("rag_vector_cache").set({
            "documents_count": len(rag_system.collection.ids),
            "embedding_model": "all-MiniLM-L6-v2",
            "updated_at": firestore.SERVER_TIMESTAMP
        })

    def load_rag_cache(self, rag_system, source_documents):
        snapshots = self.db.collection("rag_vector_cache").stream()
        rows = []

        for snapshot in snapshots:
            data = snapshot.to_dict()
            if data.get("embedding") is not None:
                rows.append(data)

        if not rows:
            return False

        rows.sort(key=lambda row: row.get("doc_id", ""))
        rag_system.source_documents = source_documents
        rag_system.collection = SimpleVectorStore()
        rag_system.collection.add(
            embeddings=[row["embedding"] for row in rows],
            documents=[row.get("searchable_text", "") for row in rows],
            metadatas=[row.get("metadata", {}) for row in rows],
            ids=[row["doc_id"] for row in rows]
        )
        return True

    # ---------- IoT sensor readings ----------
    def save_sensor_readings(self, sensor_df, source="Unknown", feed="json"):
        if sensor_df is None or sensor_df.empty:
            return 0

        batch = self.db.batch()
        saved = 0

        for _, row in sensor_df.iterrows():
            record = {column: _to_plain_value(value) for column, value in row.items()}
            timestamp = record.get("timestamp")

            if isinstance(timestamp, str):
                timestamp = pd.to_datetime(timestamp).to_pydatetime()
            elif isinstance(timestamp, pd.Timestamp):
                timestamp = timestamp.to_pydatetime()

            record["timestamp"] = timestamp
            record["source"] = source
            record["feed"] = record.get("feed", feed)
            record["saved_at"] = firestore.SERVER_TIMESTAMP

            unique_text = f"{record.get('feed')}|{timestamp}|{record.get('value')}|" \
                          f"{record.get('temperature_c')}|{record.get('soil_moisture_pct')}"
            doc_id = hashlib.sha1(unique_text.encode("utf-8")).hexdigest()

            ref = self.db.collection("sensor_readings").document(doc_id)
            batch.set(ref, record, merge=True)
            saved += 1

            if saved % 400 == 0:
                batch.commit()
                batch = self.db.batch()

        batch.commit()

        self.db.collection("system_metadata").document("sensor_readings").set({
            "last_source": source,
            "last_feed": feed,
            "last_saved_count": saved,
            "updated_at": firestore.SERVER_TIMESTAMP
        }, merge=True)

        return saved

    def load_sensor_readings(self, limit=100, feed="json"):
        # Read recent rows and filter in Python to avoid requiring a composite index.
        snapshots = (
            self.db.collection("sensor_readings")
            .order_by("timestamp", direction=firestore.Query.DESCENDING)
            .limit(max(int(limit) * 5, 100))
            .stream()
        )

        rows = []
        for snapshot in snapshots:
            row = snapshot.to_dict()
            row.pop("saved_at", None)

            if feed != "json" and row.get("feed") != feed:
                continue
            if feed == "json" and "temperature_c" not in row:
                continue

            rows.append(row)
            if len(rows) >= int(limit):
                break

        if not rows:
            return pd.DataFrame()

        df = pd.DataFrame(rows)
        if "timestamp" in df.columns:
            df["timestamp"] = pd.to_datetime(df["timestamp"])
            df = df.sort_values("timestamp").reset_index(drop=True)

        # For the combined dashboard, keep only the combined IoT schema.
        if feed == "json":
            required = [
                "timestamp", "temperature_c", "air_humidity_pct",
                "soil_moisture_pct", "light_lux"
            ]
            available = [column for column in required if column in df.columns]
            if len(available) == len(required):
                df = df[required]

        return df

    # ---------- Plant health history ----------
    def save_health_entry(self, entry):
        payload = _to_plain_value(entry)
        payload["created_at"] = firestore.SERVER_TIMESTAMP
        self.db.collection("plant_health_history").add(payload)

    def load_health_history(self, plant_name=None, limit=200):
        query = self.db.collection("plant_health_history")
        if plant_name:
            query = query.where("plant", "==", plant_name)

        snapshots = query.limit(int(limit)).stream()
        rows = []
        for snapshot in snapshots:
            row = snapshot.to_dict()
            row.pop("created_at", None)
            rows.append(row)

        rows.sort(key=lambda row: str(row.get("timestamp", "")))
        return rows

    def replace_demo_health_history(self, plant_name, entries):
        # One-field query avoids a composite-index requirement.
        existing = (
            self.db.collection("plant_health_history")
            .where("plant", "==", plant_name)
            .stream()
        )
        batch = self.db.batch()
        for snapshot in existing:
            if snapshot.to_dict().get("is_demo") is True:
                batch.delete(snapshot.reference)
        batch.commit()

        batch = self.db.batch()
        for entry in entries:
            ref = self.db.collection("plant_health_history").document()
            payload = _to_plain_value(entry)
            payload["is_demo"] = True
            payload["created_at"] = firestore.SERVER_TIMESTAMP
            batch.set(ref, payload)
        batch.commit()

    # ---------- Watering reminders ----------
    def save_watering_reminder(self, reminder):
        plant_key = re.sub(r"[^a-zA-Z0-9_-]+", "_", reminder["plant"]).strip("_").lower()
        doc_id = plant_key or "my_plant"
        payload = _to_plain_value(reminder)
        payload["updated_at"] = firestore.SERVER_TIMESTAMP
        self.db.collection("watering_reminders").document(doc_id).set(payload)

    def load_watering_reminders(self):
        return [snapshot.to_dict() for snapshot in self.db.collection("watering_reminders").stream()]

    # ---------- Image analysis history ----------
    def save_image_analysis(self, result):
        payload = _to_plain_value(result)
        payload["created_at"] = firestore.SERVER_TIMESTAMP
        self.db.collection("image_analysis_history").add(payload)


    # ---------- AI chatbot history ----------
    def save_chat_message(self, session_id, role, content, sources=None):
        """Persists one chatbot message in Firestore."""
        payload = {
            "session_id": str(session_id or "default_session"),
            "role": str(role),
            "content": str(content),
            "sources": _to_plain_value(sources or []),
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "created_at": firestore.SERVER_TIMESTAMP,
        }
        self.db.collection("chat_messages").add(payload)

    def load_chat_history(self, session_id, limit=50):
        """Loads a chat session and returns Gradio message dictionaries."""
        snapshots = (
            self.db.collection("chat_messages")
            .where("session_id", "==", str(session_id or "default_session"))
            .limit(int(limit))
            .stream()
        )

        rows = []
        for snapshot in snapshots:
            row = snapshot.to_dict()
            rows.append(row)

        # Sort locally so Firestore does not require a composite index.
        rows.sort(key=lambda row: str(row.get("timestamp", "")))

        return [
            {
                "role": row.get("role", "assistant"),
                "content": row.get("content", ""),
            }
            for row in rows
            if row.get("content")
        ]

    def clear_chat_history(self, session_id):
        """Deletes all messages belonging to one chat session."""
        snapshots = list(
            self.db.collection("chat_messages")
            .where("session_id", "==", str(session_id or "default_session"))
            .stream()
        )

        # Commit in chunks to stay below Firestore batch limits.
        deleted = 0
        for start in range(0, len(snapshots), 400):
            batch = self.db.batch()
            for snapshot in snapshots[start:start + 400]:
                batch.delete(snapshot.reference)
                deleted += 1
            batch.commit()

        return deleted


firebase_repo = FirebaseRepository(firebase_db)
print("✅ Firebase repository initialized")


In [ ]:
# Firestore collections are created automatically.
print("✅ Persistence layer ready: papers, index, RAG cache, IoT, health history, reminders, image analysis, chatbot history")


In [ ]:
# Load research papers from Firestore first.
# Web scraping runs only once, when Firestore does not yet contain the papers.

stored_documents = firebase_repo.load_documents()

if stored_documents:
    documents = stored_documents
    print(f"✅ Loaded {len(documents)} research papers from Firebase")
else:
    print("ℹ️ No research papers found in Firebase. Scraping the sources once...")
    documents = enrich_documents_from_web(documents)
    firebase_repo.save_documents(documents)
    print(f"✅ Saved {len(documents)} research papers to Firebase")


In [ ]:
# RAG classes — semantic retrieval + Gemini answer generation

class SimpleVectorStore:
    """In-memory vector store that supports cosine similarity search."""

    def __init__(self):
        self.documents = []
        self.embeddings = []
        self.metadatas = []
        self.ids = []

    def add(self, embeddings, documents, metadatas, ids):
        self.embeddings.extend(embeddings)
        self.documents.extend(documents)
        self.metadatas.extend(metadatas)
        self.ids.extend(ids)
        print(f"✅ Added {len(documents)} documents to vector store")

    def query(self, query_embeddings, n_results=3):
        if not self.embeddings:
            return {
                "ids": [[]],
                "documents": [[]],
                "metadatas": [[]],
                "distances": [[]]
            }

        similarities = cosine_similarity(query_embeddings, self.embeddings)[0]
        top_indices = np.argsort(similarities)[::-1][:n_results]

        return {
            "ids": [[self.ids[i] for i in top_indices]],
            "documents": [[self.documents[i] for i in top_indices]],
            "metadatas": [[self.metadatas[i] for i in top_indices]],
            "distances": [[1 - similarities[i] for i in top_indices]]
        }


class CalendulaRAG:
    """Complete RAG pipeline: retrieve relevant papers and generate a grounded Gemini answer."""

    def __init__(self, ai_client=None):
        print("🌿 Initializing CalendulaRAG...")

        self.lemmatizer = WordNetLemmatizer()
        self.ai_client = ai_client
        self.source_documents = {}

        if TRANSFORMERS_AVAILABLE:
            try:
                self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
                self.use_transformers = True
                print("✅ SentenceTransformer loaded")
            except Exception:
                self.use_transformers = False
                self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")
                print("⚠️ Using TF-IDF fallback")
        else:
            self.use_transformers = False
            self.tfidf = TfidfVectorizer(max_features=1000, stop_words="english")
            print("⚠️ Using TF-IDF")

        self.collection = SimpleVectorStore()
        self.fitted = False
        print("🎉 CalendulaRAG ready!")

    def _preprocess(self, text):
        if not text:
            return ""

        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"[^\w\s\-\.\(\)]", " ", text)
        return text.strip().lower()

    def _lemmatize(self, text):
        tokens = word_tokenize(text)
        return " ".join(
            self.lemmatizer.lemmatize(word)
            for word in tokens
            if word.isalpha()
        )

    def _embed(self, texts):
        if self.use_transformers:
            return self.embedding_model.encode(texts, show_progress_bar=False)

        if not self.fitted:
            self.tfidf.fit(texts)
            self.fitted = True

        return self.tfidf.transform(texts).toarray()

    def load_papers(self, docs):
        """Stores original papers and creates embeddings once."""
        print(f"📚 Loading {len(docs)} papers...")

        self.source_documents = docs
        document_texts, metadata_list, ids = [], [], []

        for doc_id, paper in docs.items():
            searchable_text = self._lemmatize(
                self._preprocess(
                    f"{paper.get('Title', '')} "
                    f"{paper.get('abstract', '')} "
                    f"{paper.get('content', '')}"
                )
            )

            document_texts.append(searchable_text)
            ids.append(doc_id)
            metadata_list.append({
                "title": paper.get("Title", ""),
                "authors": paper.get("authors", ""),
                "journal": paper.get("journal", ""),
                "year": str(paper.get("year", "")),
                "url": paper.get("URL", ""),
                "topic": paper.get("topic", "")
            })

        embeddings = list(self._embed(document_texts))
        self.collection.add(embeddings, document_texts, metadata_list, ids)

    def search(self, query, n_results=3):
        """Semantic retrieval stage."""
        cleaned_query = self._lemmatize(self._preprocess(query))
        query_embedding = self._embed([cleaned_query])
        return self.collection.query(query_embedding, n_results)

    def _build_context(self, search_results, max_chars_per_doc=1200):
        """Builds a small context to keep Gemini fast and token-efficient."""
        context_parts = []
        sources = []

        ids = search_results["ids"][0]
        metas = search_results["metadatas"][0]
        distances = search_results["distances"][0]

        for number, (doc_id, meta, distance) in enumerate(
            zip(ids, metas, distances),
            start=1
        ):
            paper = self.source_documents.get(doc_id, {})
            abstract = str(paper.get("abstract", "")).strip()
            content = str(paper.get("content", "")).strip()
            source_text = abstract or content
            source_text = source_text[:max_chars_per_doc]

            similarity = round(1 - float(distance), 2)

            context_parts.append(
                f"""[Source {number}]
Title: {meta.get('title', '')}
Authors: {meta.get('authors', '')}
Journal: {meta.get('journal', '')} ({meta.get('year', '')})
URL: {meta.get('url', '')}
Similarity: {similarity}
Text: {source_text}"""
            )

            sources.append({
                "number": number,
                "doc_id": doc_id,
                "title": meta.get("title", ""),
                "url": meta.get("url", ""),
                "similarity": similarity
            })

        return "\n\n".join(context_parts), sources

    def _generate_answer(self, question, context):
        """Generation stage: Gemini answers only from retrieved sources."""
        prompt = f"""
You are an academic assistant for a Calendula officinalis research system.

Answer the user's question using ONLY the retrieved sources below.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts.
3. If the sources are insufficient, say that clearly.
4. Keep the answer concise and easy to understand.
5. Cite claims using [Source 1], [Source 2], etc.
6. Answer in the same language as the user's question.

USER QUESTION:
{question}

RETRIEVED SOURCES:
{context}

ANSWER:
""".strip()

        return generate_with_retry(self.ai_client, prompt)

    def query(self, question, n_results=3):
        """Runs retrieval, context building, and AI generation."""
        search_results = self.search(question, n_results)

        docs = search_results["documents"][0]
        distances = search_results["distances"][0]

        if not docs:
            return {
                "response": "⚠️ No relevant documents found.",
                "papers_found": 0,
                "confidence": 0.0,
                "source": "NO_RESULTS"
            }

        confidence = round(1 - float(distances[0]), 2)
        context, sources = self._build_context(search_results)

        try:
            ai_answer = self._generate_answer(question, context)
            generation_source = "GEMINI_RAG"
        except Exception as e:
            ai_answer = (
                "⚠️ Gemini could not generate an answer right now. "
                "The retrieved academic sources are still shown below.\n"
                f"Technical details: {e}"
            )
            generation_source = "RETRIEVAL_ONLY"

        source_lines = []
        for source in sources:
            source_lines.append(
                f"[Source {source['number']}] {source['title']}\n"
                f"Similarity: {source['similarity']}\n"
                f"URL: {source['url']}"
            )

        final_response = (
            f"🤖 AI Answer\n"
            f"{'=' * 60}\n"
            f"{ai_answer}\n\n"
            f"📚 Retrieved Sources\n"
            f"{'=' * 60}\n"
            f"{chr(10).join(source_lines)}\n\n"
            f"📊 Retrieval confidence: {confidence}"
        )

        return {
            "response": final_response,
            "papers_found": len(docs),
            "confidence": confidence,
            "source": generation_source
        }


print("✅ Complete Gemini RAG classes defined!")


In [ ]:
# Initialize search services.
# The inverted index is loaded from Firebase; it is built only if no saved index exists.

index_service = IndexService()
query_service = QueryService(index_service)
result_service = ResultService(index_service, query_service)

index_loaded = firebase_repo.load_inverted_index(index_service, documents)

if index_loaded:
    print(f"✅ Loaded inverted index from Firebase ({len(index_service.index)} terms)")
else:
    print("ℹ️ No inverted index found in Firebase. Building it once...")
    for doc_id, doc_data in documents.items():
        index_service.add_document(doc_id, doc_data)

    firebase_repo.save_inverted_index(index_service)
    print(f"✅ Built and saved inverted index ({len(index_service.index)} terms)")

# Required assignment table: term + DocIDs
index_df = index_service.to_dataframe()
display(index_df.head(30))

print("\nLemmatization examples:")
for word in ["healing", "flowers", "treatments", "pathogens", "studies"]:
    print(f"  {word:<15} -> {index_service.lemmatize_word(word)}")


In [ ]:
# Initialize semantic RAG.
# Saved document embeddings are loaded from Firebase and are created only once.

rag_system = CalendulaRAG(ai_client=gemini_client)

rag_cache_loaded = firebase_repo.load_rag_cache(rag_system, documents)

if rag_cache_loaded:
    print(f"✅ Loaded RAG vector cache from Firebase ({len(rag_system.collection.ids)} papers)")
else:
    print("ℹ️ No RAG vector cache found in Firebase. Creating embeddings once...")
    rag_system.load_papers(documents)
    firebase_repo.save_rag_cache(rag_system)
    print("✅ RAG vector cache saved to Firebase")

print("✅ Search systems are ready")


In [ ]:
# Gradio UI helper functions
# Defines all functions used by the Gradio tabs
# Gradio web interface - main app with tabs

# This version uses Gradio as the main UI and places every feature
# inside one interface: Image Upload, IoT Sensors, Search, Dashboard,
# and Plant Health History.

# --- Screen 1: Image analysis ---
PLANT_IMAGE_STATUS = "Unknown"
IMAGE_VALIDATION_PASSED = False


def validate_plant_image_with_ai(pil_image):
    """Uses Gemini vision to verify that the uploaded image contains a plant or flower."""
    if pil_image is None:
        return False, "Please upload an image first."

    if gemini_client is None:
        return False, "AI image validation is unavailable because Gemini is not connected."

    try:
        image_buffer = io.BytesIO()
        pil_image.convert("RGB").save(image_buffer, format="JPEG", quality=90)
        image_part = types.Part.from_bytes(
            data=image_buffer.getvalue(),
            mime_type="image/jpeg"
        )

        prompt = """
Inspect the uploaded image and decide whether it visibly contains a real plant, flower,
leaf, stem, potted plant, or another clearly identifiable botanical subject.

Return exactly one word:
PLANT - if a plant or flower is clearly visible.
NOT_PLANT - if no plant or flower is visible, or the image is too unclear to verify.
""".strip()

        last_error = None
        for attempt in range(4):
            try:
                response = gemini_client.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=[prompt, image_part],
                    config=types.GenerateContentConfig(
                        temperature=0.0,
                        max_output_tokens=10
                    )
                )
                answer = str(getattr(response, "text", "")).strip().upper()

                if answer == "PLANT":
                    return True, "AI verified that the image contains a plant or flower."
                if answer == "NOT_PLANT":
                    return False, "The uploaded image does not appear to contain a plant or flower."

                return False, "AI could not confidently verify that the image contains a plant or flower."

            except (errors.ServerError, errors.ClientError) as error:
                last_error = error
                status_code = getattr(error, "code", None)
                if status_code not in (429, 503) or attempt == 3:
                    raise
                time.sleep(2 ** attempt)

        raise last_error

    except Exception as error:
        print(f"Image validation warning: {error}")
        return False, "AI image validation failed. Please try again in a moment."


def analyze_plant_image(pil_image):
    """First verifies the image with AI, then runs the existing RGB health algorithm."""
    global PLANT_IMAGE_STATUS, IMAGE_VALIDATION_PASSED

    IMAGE_VALIDATION_PASSED = False

    if pil_image is None:
        return None, "❌ Please upload a plant or flower image first."

    is_plant, validation_message = validate_plant_image_with_ai(pil_image)

    if not is_plant:
        PLANT_IMAGE_STATUS = "Unknown"
        return None, (
            "## ❌ Image rejected\n\n"
            f"{validation_message}\n\n"
            "Please upload a clear photo that contains a plant or flower. "
            "The health analysis was not performed."
        )

    img = np.array(pil_image.convert("RGB"))
    r, g, b = img[:, :, 0].mean(), img[:, :, 1].mean(), img[:, :, 2].mean()

    if (g > r + 15 and g > b + 10) or (r > 80 and g > 80 and b < 80 and abs(r - g) < 60):
        status = "Healthy"
        advice = "Plant looks good! Keep the current watering schedule."
        emoji = "✅"
    elif r > g + 20 and r > 100 and b < 80:
        status = "Dry"
        advice = "The plant may be dry. Consider watering soon."
        emoji = "💧"
    else:
        status = "Sick"
        advice = "Unusual colors were detected. Check for pests, disease, or over-watering."
        emoji = "⚠️"

    PLANT_IMAGE_STATUS = status
    IMAGE_VALIDATION_PASSED = True

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor(COLORS["bg"])
    axes[0].imshow(pil_image)
    axes[0].set_title("Uploaded Plant Image", fontsize=14, color=COLORS["text"], fontweight="bold")
    axes[0].axis("off")

    axes[1].set_facecolor(COLORS["light"])
    axes[1].axis("off")
    axes[1].text(
        0.5, 0.84, "🤖 AI check: Plant verified", ha="center", fontsize=13,
        fontweight="bold", color=COLORS["primary"], transform=axes[1].transAxes
    )
    axes[1].text(
        0.5, 0.68, f"{emoji} Status: {status}", ha="center", fontsize=22,
        fontweight="bold", color=COLORS.get(status.lower(), COLORS["primary"]),
        transform=axes[1].transAxes
    )
    axes[1].text(
        0.5, 0.48, f"Average RGB: ({round(r, 1)}, {round(g, 1)}, {round(b, 1)})",
        ha="center", fontsize=13, color=COLORS["text"], transform=axes[1].transAxes
    )
    axes[1].text(
        0.5, 0.28, advice, ha="center", fontsize=12, color=COLORS["text"],
        wrap=True, transform=axes[1].transAxes
    )
    axes[1].set_title(
        "OrangePanther Analysis Result", fontsize=14,
        color=COLORS["text"], fontweight="bold"
    )
    plt.tight_layout()

    image_result = {
        "ai_verified_plant": True,
        "status": status,
        "advice": advice,
        "average_r": round(float(r), 1),
        "average_g": round(float(g), 1),
        "average_b": round(float(b), 1),
        "analyzed_at": datetime.now(timezone.utc),
    }
    firebase_repo.save_image_analysis(image_result)

    summary = (
        "**AI verification:** ✅ Plant or flower detected\n\n"
        f"**Status:** {status}\n\n"
        f"**Advice:** {advice}\n\n"
        f"**Average RGB:** ({round(r, 1)}, {round(g, 1)}, {round(b, 1)})\n\n"
        "☁️ Valid analysis saved to Firebase."
    )
    return fig, summary


# --- Screen 2: IoT sensor data ---
BASE_URL = "https://server-cloud-v645.onrender.com/"


def fetch_sensor_df(feed="json", limit=30):
    """Fetches sensor readings from the cloud server.

    feed options:
    - "json": returns combined sensor readings and is used by the dashboard.
    - "humidity", "soil", "temperature": returns one selected sensor feed.
    """
    try:
        response = requests.get(
            f"{BASE_URL}/history",
            params={"feed": feed, "limit": int(limit)},
            timeout=60
        )
        data = response.json()

        if "data" not in data:
            return None

        # Combined feed: used for dashboard and 2x2 sensor visualization
        if feed == "json":
            rows = []
            for sample in data["data"]:
                try:
                    values = json_lib.loads(sample["value"])
                    rows.append({
                        "timestamp": pd.to_datetime(sample["created_at"]),
                        "temperature_c": float(values.get("temperature", 0)),
                        "air_humidity_pct": float(values.get("humidity", 0)),
                        "soil_moisture_pct": float(str(values.get("soil", 0)).strip()),
                        "light_lux": 600.0,
                    })
                except Exception:
                    continue
            if not rows:
                return None
            return pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)

        # Single-feed view: matches the lecturer examples from the Render server.
        rows = []
        for sample in data["data"]:
            try:
                rows.append({
                    "timestamp": pd.to_datetime(sample["created_at"]),
                    "feed": feed,
                    "value": pd.to_numeric(sample["value"], errors="coerce"),
                    "raw_value": sample["value"],
                })
            except Exception:
                continue
        if not rows:
            return None
        return pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)

    except Exception:
        return None

def generate_sensor_data(hours=24, interval_minutes=30):
    """Fallback: generates simulated sensor readings with daily patterns."""
    n_points = int(hours * 60 / interval_minutes)
    start = datetime.now() - timedelta(hours=hours)
    timestamps = [start + timedelta(minutes=i * interval_minutes) for i in range(n_points)]
    data = []
    temp, humidity, soil, light = 24.0, 55.0, 45.0, 600.0
    for ts in timestamps:
        hour = ts.hour + ts.minute / 60
        day_factor = max(0, np.sin((hour - 6) * np.pi / 12))
        temp = float(np.clip(temp + random.uniform(-0.8, 0.8) + (day_factor - 0.3) * 0.3, 18, 32))
        humidity = float(np.clip(humidity + random.uniform(-2, 2) - (day_factor - 0.3) * 1.5, 35, 85))
        soil = float(np.clip(soil + random.uniform(-1.5, 1.5), 15, 75))
        light = float(np.clip(200 + day_factor * 900 + random.uniform(-80, 80), 50, 1300))
        data.append({
            "timestamp": ts,
            "temperature_c": round(temp, 1),
            "air_humidity_pct": round(humidity, 1),
            "soil_moisture_pct": round(soil, 1),
            "light_lux": round(light, 0),
        })
    return pd.DataFrame(data)


# Database-first startup:
# use saved IoT data and contact the external server only when Firebase is empty.
sensor_df = firebase_repo.load_sensor_readings(limit=100, feed="json")

if not sensor_df.empty:
    data_source = "Firebase"
    print(f"✅ Loaded {len(sensor_df)} IoT readings from Firebase")
else:
    _result = fetch_sensor_df(feed="json", limit=30)

    if _result is not None:
        sensor_df = _result
        data_source = "Live"
        firebase_repo.save_sensor_readings(sensor_df, source="Live", feed="json")
        print("✅ Initial live IoT data saved to Firebase")
    else:
        sensor_df = generate_sensor_data()
        data_source = "Simulated"
        firebase_repo.save_sensor_readings(sensor_df, source="Simulated", feed="json")
        print("⚠️ No cloud data was available. Simulated fallback saved to Firebase")


def plot_sensor_data(df):
    """Creates a sensor trend plot.

    If the data contains one selected feed, a single chart is shown.
    If the data contains the combined json feed, a 2x2 dashboard chart is shown.
    """
    if "value" in df.columns:
        fig, ax = plt.subplots(figsize=(12, 4))
        fig.patch.set_facecolor(COLORS["bg"])
        ax.set_facecolor(COLORS["light"])
        feed_name = df["feed"].iloc[0] if "feed" in df.columns and len(df) else "selected feed"
        ax.plot(df["timestamp"], df["value"], color=COLORS["primary"], linewidth=2, marker="o", markersize=4)
        ax.fill_between(df["timestamp"], df["value"], alpha=0.25, color=COLORS["secondary"])
        ax.set_title(f"OrangePanther - {feed_name.capitalize()} Feed", fontsize=14, fontweight="bold", color=COLORS["text"])
        ax.set_xlabel("Time")
        ax.set_ylabel("Value")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        return fig

    fig, axes = plt.subplots(2, 2, figsize=(14, 7))
    fig.patch.set_facecolor(COLORS["bg"])
    fig.suptitle("OrangePanther - IoT Sensor Readings", fontsize=14, fontweight="bold", color=COLORS["text"])
    for ax, (col, title, color) in zip(axes.flat, [
        ("temperature_c", "Temperature (C)", COLORS["accent"]),
        ("air_humidity_pct", "Air Humidity (%)", COLORS["primary"]),
        ("soil_moisture_pct", "Soil Moisture (%)", COLORS["brown"]),
        ("light_lux", "Light Intensity (lux)", COLORS["secondary"]),
    ]):
        ax.set_facecolor(COLORS["light"])
        ax.fill_between(df["timestamp"], df[col], alpha=0.35, color=color)
        ax.plot(df["timestamp"], df[col], color=color, linewidth=2, marker="o", markersize=3)
        ax.set_title(title, color=COLORS["text"], fontweight="bold")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    return fig

def refresh_sensor_data(feed, limit):
    """Fetches fresh IoT data, saves it, and falls back to Firestore when needed."""
    global sensor_df, data_source

    limit = int(limit)
    new_df = fetch_sensor_df(feed=feed, limit=limit)

    if new_df is not None and not new_df.empty:
        saved_count = firebase_repo.save_sensor_readings(
            new_df,
            source="Live",
            feed=feed
        )
        data_source = "Live"

        if feed == "json":
            sensor_df = new_df

        message = (
            f"✅ Live data loaded and saved to Firebase. "
            f"Feed: {feed} | Rows: {len(new_df)} | Saved: {saved_count}"
        )
        return message, new_df.tail(10), plot_sensor_data(new_df)

    # The server is unavailable: read the latest saved values instead.
    stored_df = firebase_repo.load_sensor_readings(limit=limit, feed=feed)

    if not stored_df.empty:
        data_source = "Firebase"
        if feed == "json":
            sensor_df = stored_df

        message = (
            f"☁️ Cloud server unavailable. Loaded {len(stored_df)} saved "
            f"rows from Firebase instead."
        )
        return message, stored_df.tail(10), plot_sensor_data(stored_df)

    # Last resort only: create fallback data once and persist it.
    fallback_df = generate_sensor_data()
    firebase_repo.save_sensor_readings(
        fallback_df,
        source="Simulated",
        feed="json"
    )
    sensor_df = fallback_df
    data_source = "Simulated"

    message = (
        "⚠️ No live or saved IoT data was found. "
        "Simulated fallback data was created and saved to Firebase."
    )
    return message, fallback_df.tail(10), plot_sensor_data(fallback_df)


# --- Screen 3: Search functions ---
def sim_to_simple(sim):
    """Converts a similarity score into a simple user-friendly label."""
    if sim > 0.4:
        return "High match"
    elif sim > 0.2:
        return "Medium match"
    return "Low match"


def run_gradio_search(question, search_type, n_results=3, show_advanced=False):
    """Routes the question to complete AI-RAG or exact Inverted Index search."""
    if not question or not question.strip():
        return "Please enter a question."

    if search_type == "RAG (semantic)":
        result = rag_system.query(
            question=question,
            n_results=int(n_results)
        )

        output = result["response"]

        if show_advanced:
            output += (
                f"\n\n🔧 Advanced details"
                f"\nPapers found: {result['papers_found']}"
                f"\nConfidence: {result['confidence']}"
                f"\nGeneration source: {result['source']}"
                f"\nEmbedding model: all-MiniLM-L6-v2"
                f"\nAI model: {GEMINI_MODEL}"
            )

        return output

    # Exact inverted-index search
    terms = question.lower().replace("-", " ").split()
    query = query_service.create_query(terms)
    results = query["results"]

    if not results:
        return f"No results found for: {terms}"

    lines = [
        f"Inverted Index results for: {terms}",
        "-" * 50
    ]

    for doc_id in sorted(results):
        doc = index_service.get_document(doc_id)
        lines.append(f"Paper: {doc_id} - {doc['Title']}")
        lines.append(f"URL: {doc['URL']}")

        if show_advanced:
            lines.append(
                f"Advanced details: matched by exact index query terms = {terms}"
            )

        lines.append("")

    return "\n".join(lines)


# --- Screen 4: Dashboard ---
def get_status_label(col, value):
    """Returns a Normal / Warning / Danger label with color for a sensor reading."""
    thresholds = {
        "temperature_c": [(18, 28, "Normal", "#27AE60"), (28, 30, "Warning", "#E67E22"), (30, 99, "High", "#E74C3C")],
        "soil_moisture_pct": [(30, 70, "Normal", "#27AE60"), (20, 30, "Low", "#E67E22"), (0, 20, "Danger", "#E74C3C")],
        "light_lux": [(400, 9999, "Normal", "#27AE60"), (300, 400, "Low Light", "#E67E22"), (0, 300, "Too Dark", "#E74C3C")],
        "air_humidity_pct": [(40, 80, "Normal", "#27AE60"), (30, 40, "Dry", "#E67E22"), (0, 30, "Very Dry", "#E74C3C")],
    }
    for mn, mx, label, color in thresholds.get(col, []):
        if mn <= value < mx:
            return label, color
    return "Normal", "#27AE60"


def dashboard_status(plant_name):
    """Builds a dashboard summary and sensor trend figure."""
    plant_name = plant_name.strip() or "My Plant"
    latest = sensor_df.iloc[-1]
    issues, recs = [], []

    if latest["soil_moisture_pct"] < 30:
        issues.append("Needs Water")
        recs.append(f"{plant_name} needs water.")
    if latest["temperature_c"] > 30:
        issues.append("Too Hot")
        recs.append(f"{plant_name} is in a hot environment. Move it to a cooler place.")
    if latest["light_lux"] < 300:
        issues.append("Low Light")
        recs.append(f"{plant_name} needs more light.")
    if latest["air_humidity_pct"] < 40:
        recs.append(f"The air around {plant_name} is dry. Consider misting.")
    if PLANT_IMAGE_STATUS == "Sick":
        issues.append("Possible Disease")
        recs.append(f"Image analysis suggests that {plant_name} may have a disease.")
    if PLANT_IMAGE_STATUS == "Dry" and "Needs Water" not in issues:
        issues.append("Needs Water")
        recs.append(f"Image analysis suggests that {plant_name} needs water.")

    overall = issues[0] if issues else "Healthy"
    if not recs:
        recs = [f"{plant_name} looks great. Keep up the current care routine."]

    labels = []
    for label, col, unit in [
        ("Temperature", "temperature_c", "C"),
        ("Soil Moisture", "soil_moisture_pct", "%"),
        ("Light", "light_lux", "lux"),
        ("Air Humidity", "air_humidity_pct", "%"),
    ]:
        status_label, _ = get_status_label(col, latest[col])
        labels.append(f"- **{label}:** {latest[col]} {unit} ({status_label})")

    markdown = f"""
# OrangePanther Dashboard

**Plant:** {plant_name}
**Last updated:** {latest['timestamp'].strftime('%Y-%m-%d %H:%M')}
**Overall status:** {overall}
**Image analysis status:** {PLANT_IMAGE_STATUS}

## Current sensor values
{chr(10).join(labels)}

## Care tips
{chr(10).join('- ' + r for r in recs)}
"""
    return markdown, plot_sensor_data(sensor_df.tail(24))


# --- Screen 5: Plant Health History as a Gradio tab ---
STATUS_SCORE = {
    "Healthy": 3, "Needs Water": 2, "Low Light": 2,
    "Too Hot": 1, "Possible Disease": 0, "Unknown": 2,
}
STATUS_COLOR_MAP = {
    "Healthy": "#27AE60", "Needs Water": "#E67E22", "Low Light": "#E67E22",
    "Too Hot": "#E74C3C", "Possible Disease": "#C0392B", "Unknown": "#95A5A6",
}
HEALTH_HISTORY = firebase_repo.load_health_history(limit=200)
print(f"✅ Loaded {len(HEALTH_HISTORY)} plant health entries from Firebase")


def determine_current_status():
    """Determines the plant status from the latest sensor row and image status."""
    latest = sensor_df.iloc[-1]
    if PLANT_IMAGE_STATUS == "Sick":
        return "Possible Disease"
    if latest["soil_moisture_pct"] < 30 or PLANT_IMAGE_STATUS == "Dry":
        return "Needs Water"
    if latest["temperature_c"] > 30:
        return "Too Hot"
    if latest["light_lux"] < 300:
        return "Low Light"
    return "Healthy"


def log_health_entry(plant_name):
    """Saves a timestamped health check-in to Firebase."""
    global HEALTH_HISTORY

    plant_name = plant_name.strip() or "My Plant"
    latest = sensor_df.iloc[-1]
    status = determine_current_status()

    entry = {
        "timestamp": datetime.now(timezone.utc),
        "plant": plant_name,
        "status": status,
        "score": STATUS_SCORE.get(status, 2),
        "temperature": float(latest["temperature_c"]),
        "soil": float(latest["soil_moisture_pct"]),
        "light": float(latest["light_lux"]),
        "humidity": float(latest["air_humidity_pct"]),
        "is_demo": False,
    }

    firebase_repo.save_health_entry(entry)
    HEALTH_HISTORY = firebase_repo.load_health_history(limit=200)

    message = (
        f"✅ Check-in saved to Firebase for {plant_name}. "
        f"Status: {status}. Total entries: {len(HEALTH_HISTORY)}"
    )
    table, fig, summary = get_health_history_outputs(plant_name)
    return message, table, fig, summary


def seed_demo_history(plant_name="Calendula"):
    """Creates demo entries once in Firebase and reloads the saved history."""
    global HEALTH_HISTORY

    plant_name = plant_name.strip() or "Calendula"
    statuses = [
        "Healthy", "Healthy", "Needs Water", "Needs Water", "Too Hot",
        "Healthy", "Low Light", "Healthy", "Needs Water", "Healthy"
    ]
    base = {
        "temperature": [24, 25, 26, 27, 32, 25, 24, 23, 26, 25],
        "soil": [55, 50, 28, 22, 35, 52, 48, 60, 25, 58],
        "light": [700, 650, 600, 580, 500, 680, 250, 720, 600, 710],
        "humidity": [55, 52, 48, 45, 40, 58, 55, 60, 50, 56],
    }

    entries = []
    for i, status in enumerate(statuses):
        timestamp = datetime.now(timezone.utc) - timedelta(
            days=9 - i,
            hours=random.randint(8, 20)
        )
        entries.append({
            "timestamp": timestamp,
            "plant": plant_name,
            "status": status,
            "score": STATUS_SCORE.get(status, 2),
            "temperature": base["temperature"][i],
            "soil": base["soil"][i],
            "light": base["light"][i],
            "humidity": base["humidity"][i],
            "is_demo": True,
        })

    firebase_repo.replace_demo_health_history(plant_name, entries)
    HEALTH_HISTORY = firebase_repo.load_health_history(limit=200)

    table, fig, summary = get_health_history_outputs(plant_name)
    return (
        f"🎭 Demo data saved to Firebase for {plant_name}.",
        table,
        fig,
        summary
    )


def get_health_history_outputs(plant_name):
    """Loads saved history from Firebase and returns table, plot, and summary."""
    global HEALTH_HISTORY

    plant_name = plant_name.strip() or "My Plant"
    HEALTH_HISTORY = firebase_repo.load_health_history(limit=200)
    plant_history = [e for e in HEALTH_HISTORY if e.get("plant") == plant_name]
    if not plant_history:
        empty_df = pd.DataFrame(columns=["timestamp", "plant", "status", "temperature", "soil", "light", "humidity"])
        return empty_df, None, "No history yet. Click **Log Check-in** or **Load Demo**."

    df = pd.DataFrame(plant_history)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.sort_values("timestamp").reset_index(drop=True)

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fig.patch.set_facecolor(COLORS["bg"])
    fig.suptitle(f"Health History — {plant_name}", fontsize=15, fontweight="bold", color=COLORS["text"])

    scores = df["score"].tolist()
    statuses = df["status"].tolist()
    colors_bar = [STATUS_COLOR_MAP.get(s, "#95A5A6") for s in statuses]

    axes[0].set_facecolor(COLORS["light"])
    axes[0].plot(range(len(scores)), scores, color=COLORS["brown"], linewidth=1.5, linestyle="--", alpha=0.5)
    for i, (score, color, status) in enumerate(zip(scores, colors_bar, statuses)):
        axes[0].scatter(i, score, color=color, s=100)
        axes[0].annotate(status, (i, score), textcoords="offset points", xytext=(0, 10),
                         ha="center", fontsize=8, color=color, fontweight="bold")
    axes[0].set_yticks([0, 1, 2, 3])
    axes[0].set_yticklabels(["Danger", "Bad", "Warning", "Healthy"])
    axes[0].set_title("Plant Health Score Over Time", color=COLORS["text"], fontweight="bold")
    axes[0].grid(True, alpha=0.2)
    axes[0].set_ylim(-0.3, 3.5)

    axes[1].set_facecolor(COLORS["light"])
    axes[1].fill_between(range(len(df)), df["soil"], alpha=0.3, color=COLORS["brown"])
    axes[1].plot(range(len(df)), df["soil"], color=COLORS["brown"], linewidth=2, marker="o")
    axes[1].axhline(y=30, color="#E74C3C", linestyle=":", linewidth=1.5, alpha=0.8, label="Min threshold (30%)")
    axes[1].set_title("Soil Moisture Trend", color=COLORS["text"], fontweight="bold")
    axes[1].set_ylabel("Soil Moisture (%)")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.2)
    plt.tight_layout()

    total = len(df)
    status_counts = Counter(statuses)
    healthy_pct = round(status_counts.get("Healthy", 0) / total * 100)
    most_common = status_counts.most_common(1)[0][0]
    mid = total // 2
    first = sum(scores[:mid]) / max(mid, 1)
    second = sum(scores[mid:]) / max(total - mid, 1)
    trend = "Improving" if second > first else ("Declining" if second < first else "Stable")

    summary = f"""
**Total check-ins:** {total}
**Healthy rate:** {healthy_pct}%
**Most common status:** {most_common}
**Overall trend:** {trend}
"""
    return df.tail(8), fig, summary


# --- Watering reminders persisted in Firebase ---
WATERING_REMINDERS = firebase_repo.load_watering_reminders()


def set_watering_reminder(plant_name, every_days, reminder_time):
    """Creates or updates a persistent watering reminder in Firebase."""
    global WATERING_REMINDERS

    plant_name = plant_name.strip() or "My Plant"
    reminder = {
        "plant": plant_name,
        "every_days": int(every_days),
        "reminder_time": reminder_time,
        "created_at": datetime.now(timezone.utc),
    }

    firebase_repo.save_watering_reminder(reminder)
    WATERING_REMINDERS = firebase_repo.load_watering_reminders()

    return (
        f"✅ Watering reminder saved to Firebase for {plant_name}: "
        f"every {int(every_days)} day(s) at {reminder_time}."
    )


print(
    f"✅ Application data loaded from Firebase: "
    f"{len(HEALTH_HISTORY)} health entries, "
    f"{len(WATERING_REMINDERS)} reminders"
)


# --- Screen 6: Intelligent AI Chatbot ---
DEFAULT_CHAT_SESSION = "orange_panther_demo"


def _normalize_chat_history(history):
    """Normalizes Gradio chat history to role/content dictionaries."""
    if not history:
        return []

    normalized = []
    for item in history:
        if isinstance(item, dict):
            role = item.get("role")
            content = item.get("content")
            if role in {"user", "assistant"} and content:
                normalized.append({"role": role, "content": str(content)})
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            user_text, assistant_text = item
            if user_text:
                normalized.append({"role": "user", "content": str(user_text)})
            if assistant_text:
                normalized.append({"role": "assistant", "content": str(assistant_text)})
    return normalized


def _chat_history_as_text(history, max_messages=10):
    """Converts recent messages into a compact prompt section."""
    normalized = _normalize_chat_history(history)[-int(max_messages):]
    lines = []
    for message in normalized:
        speaker = "USER" if message["role"] == "user" else "ASSISTANT"
        lines.append(f"{speaker}: {message['content']}")
    return "\n".join(lines) if lines else "No previous conversation."


def _current_system_context():
    """Returns a short snapshot of the application's current stored state."""
    context_lines = [f"Latest image status: {PLANT_IMAGE_STATUS}"]

    try:
        if sensor_df is not None and not sensor_df.empty:
            latest = sensor_df.iloc[-1].to_dict()
            clean_latest = {
                str(key): _to_plain_value(value)
                for key, value in latest.items()
            }
            context_lines.append(f"Latest IoT reading: {clean_latest}")
        else:
            context_lines.append("Latest IoT reading: unavailable")
    except Exception:
        context_lines.append("Latest IoT reading: unavailable")

    try:
        reminders = firebase_repo.load_watering_reminders()
        if reminders:
            context_lines.append(f"Watering reminders: {reminders[:3]}")
        else:
            context_lines.append("Watering reminders: none")
    except Exception:
        context_lines.append("Watering reminders: unavailable")

    return "\n".join(context_lines)


def _retrieve_chat_sources(question, n_results=3):
    """Retrieves academic context for the chatbot using the existing RAG store."""
    try:
        search_results = rag_system.search(question, n_results=int(n_results))
        context, sources = rag_system._build_context(
            search_results,
            max_chars_per_doc=900
        )

        # Avoid grounding the response in extremely weak matches.
        relevant_sources = [
            source for source in sources
            if float(source.get("similarity", 0)) >= 0.15
        ]

        if not relevant_sources:
            return "No sufficiently relevant academic source was found.", []

        allowed_numbers = {source["number"] for source in relevant_sources}
        context_blocks = []
        for block in context.split("\n\n"):
            match = re.search(r"\[Source\s+(\d+)\]", block)
            if match and int(match.group(1)) in allowed_numbers:
                context_blocks.append(block)

        return "\n\n".join(context_blocks), relevant_sources

    except Exception as error:
        print(f"⚠️ Chat retrieval error: {error}")
        return "Academic retrieval is temporarily unavailable.", []


def chatbot_respond(message, history, session_id):
    """
    Intelligent conversational assistant:
    - remembers recent messages
    - uses current IoT/application state
    - retrieves relevant academic papers
    - generates the answer with Gemini
    - persists both sides of the conversation in Firestore
    """
    message = str(message or "").strip()
    session_id = str(session_id or DEFAULT_CHAT_SESSION).strip() or DEFAULT_CHAT_SESSION
    normalized_history = _normalize_chat_history(history)

    if not message:
        return normalized_history, "", "Please type a message."

    academic_context, sources = _retrieve_chat_sources(message, n_results=3)
    conversation_context = _chat_history_as_text(normalized_history)
    system_context = _current_system_context()

    prompt = f"""
You are OrangePanther, an intelligent AI assistant inside a Calendula plant-monitoring application.

Your responsibilities:
- Answer naturally and helpfully in the same language as the user.
- Remember the recent conversation and use it when relevant.
- For scientific claims about Calendula, prefer the retrieved academic sources and cite them as [Source 1], [Source 2], etc.
- For questions about the user's current plant, sensors, reminders, image analysis, or application state, use the CURRENT SYSTEM STATE.
- If the available information is insufficient, say so clearly instead of inventing facts.
- Do not claim to have measured or observed anything that is not present in the system state.
- Keep the response focused and easy to understand.
- Do not provide a medical diagnosis; suggest professional advice for serious health concerns.

RECENT CONVERSATION:
{conversation_context}

CURRENT SYSTEM STATE:
{system_context}

RETRIEVED ACADEMIC CONTEXT:
{academic_context}

NEW USER MESSAGE:
{message}

ASSISTANT ANSWER:
""".strip()

    try:
        answer = generate_with_retry(gemini_client, prompt, max_retries=4)
    except Exception as error:
        answer = (
            "⚠️ The AI assistant is temporarily unavailable. "
            f"Please try again shortly. Technical details: {error}"
        )

    updated_history = normalized_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]

    try:
        firebase_repo.save_chat_message(session_id, "user", message)
        firebase_repo.save_chat_message(session_id, "assistant", answer, sources=sources)
        save_status = "✅ Conversation saved to Firebase"
    except Exception as error:
        save_status = f"⚠️ Answer created, but chat history was not saved: {error}"

    return updated_history, "", save_status


def load_chat_session(session_id):
    """Loads an existing chatbot conversation from Firebase."""
    session_id = str(session_id or DEFAULT_CHAT_SESSION).strip() or DEFAULT_CHAT_SESSION
    try:
        history = firebase_repo.load_chat_history(session_id, limit=50)
        if history:
            return history, f"✅ Loaded {len(history)} messages from Firebase"
        return [], "ℹ️ No saved messages were found for this session"
    except Exception as error:
        return [], f"❌ Could not load chat history: {error}"


def clear_chat_session(session_id):
    """Clears one chatbot session from Firestore and from the UI."""
    session_id = str(session_id or DEFAULT_CHAT_SESSION).strip() or DEFAULT_CHAT_SESSION
    try:
        deleted = firebase_repo.clear_chat_history(session_id)
        return [], f"✅ Cleared {deleted} saved messages"
    except Exception as error:
        return [], f"❌ Could not clear chat history: {error}"


# -----------------------------------------------------------------------------
# Gamification: a persistent virtual plant that grows with user activity
# -----------------------------------------------------------------------------
DEFAULT_GAME_USER = "demo_user"

GAME_STAGES = [
    {"name": "Seed", "emoji": "🌰", "min_points": 0},
    {"name": "Sprout", "emoji": "🌱", "min_points": 25},
    {"name": "Young Plant", "emoji": "🌿", "min_points": 50},
    {"name": "Growing Plant", "emoji": "🪴", "min_points": 100},
    {"name": "Blooming Plant", "emoji": "🌼", "min_points": 150},
]

ACTION_POINTS = {
    "image_analysis": 10,
    "iot_refresh": 5,
    "rag_search": 8,
    "watering_reminder": 10,
    "health_checkin": 15,
    "chat_message": 5,
}


def _safe_game_user(user_id):
    return str(user_id or DEFAULT_GAME_USER).strip() or DEFAULT_GAME_USER


def _stage_for_points(points):
    points = max(int(points or 0), 0)
    current = GAME_STAGES[0]
    next_stage = None

    for index, stage in enumerate(GAME_STAGES):
        if points >= stage["min_points"]:
            current = stage
            next_stage = GAME_STAGES[index + 1] if index + 1 < len(GAME_STAGES) else None
        else:
            break

    return current, next_stage


def _default_game_profile(user_id):
    return {
        "user_id": user_id,
        "points": 0,
        "actions_count": 0,
        "stage": "Seed",
        "stage_emoji": "🌰",
    }


def load_game_profile(user_id=DEFAULT_GAME_USER):
    """Loads one persistent gamification profile from Firestore."""
    user_id = _safe_game_user(user_id)
    snapshot = firebase_repo.db.collection("gamification_profiles").document(user_id).get()

    if snapshot.exists:
        profile = snapshot.to_dict()
    else:
        profile = _default_game_profile(user_id)
        firebase_repo.db.collection("gamification_profiles").document(user_id).set({
            **profile,
            "created_at": firestore.SERVER_TIMESTAMP,
            "updated_at": firestore.SERVER_TIMESTAMP,
        })

    profile["user_id"] = user_id
    profile["points"] = int(profile.get("points", 0) or 0)
    profile["actions_count"] = int(profile.get("actions_count", 0) or 0)
    stage, _ = _stage_for_points(profile["points"])
    profile["stage"] = stage["name"]
    profile["stage_emoji"] = stage["emoji"]
    return profile


def award_game_points(action, points=None, user_id=DEFAULT_GAME_USER, details=""):
    """Adds points, stores an event, and updates the virtual plant stage."""
    user_id = _safe_game_user(user_id)
    points = int(points if points is not None else ACTION_POINTS.get(action, 0))

    if points <= 0:
        return load_game_profile(user_id)

    profile_ref = firebase_repo.db.collection("gamification_profiles").document(user_id)
    profile = load_game_profile(user_id)
    new_points = profile["points"] + points
    new_actions_count = profile["actions_count"] + 1
    stage, _ = _stage_for_points(new_points)

    profile_ref.set({
        "user_id": user_id,
        "points": new_points,
        "actions_count": new_actions_count,
        "stage": stage["name"],
        "stage_emoji": stage["emoji"],
        "last_action": action,
        "updated_at": firestore.SERVER_TIMESTAMP,
    }, merge=True)

    firebase_repo.db.collection("gamification_events").add({
        "user_id": user_id,
        "action": action,
        "points_awarded": points,
        "details": str(details or "")[:500],
        "total_points_after": new_points,
        "stage_after": stage["name"],
        "created_at": firestore.SERVER_TIMESTAMP,
    })

    return load_game_profile(user_id)


def load_game_events(user_id=DEFAULT_GAME_USER, limit=12):
    """Loads recent point events without requiring a composite Firestore index."""
    user_id = _safe_game_user(user_id)
    snapshots = firebase_repo.db.collection("gamification_events").stream()
    rows = []

    for snapshot in snapshots:
        data = snapshot.to_dict()
        if data.get("user_id") != user_id:
            continue
        created_at = data.get("created_at")
        rows.append({
            "Time": created_at,
            "Action": data.get("action", ""),
            "Points": data.get("points_awarded", 0),
            "Total": data.get("total_points_after", 0),
            "Stage": data.get("stage_after", ""),
        })

    rows.sort(key=lambda row: row.get("Time") or datetime.min.replace(tzinfo=timezone.utc), reverse=True)
    return pd.DataFrame(rows[:int(limit)])


def render_virtual_plant(profile):
    """Returns a visible plant card with stage and progress to the next stage."""
    points = int(profile.get("points", 0) or 0)
    current, next_stage = _stage_for_points(points)

    if next_stage:
        stage_span = max(next_stage["min_points"] - current["min_points"], 1)
        stage_progress = points - current["min_points"]
        progress_pct = min(max(int((stage_progress / stage_span) * 100), 0), 100)
        next_text = f"{next_stage['emoji']} {next_stage['name']} at {next_stage['min_points']} points"
        remaining = max(next_stage["min_points"] - points, 0)
        remaining_text = f"Only **{remaining} points** to the next stage."
    else:
        progress_pct = 100
        next_text = "Maximum stage reached"
        remaining_text = "Your plant is fully blooming!"

    filled = round(progress_pct / 10)
    progress_bar = "🟩" * filled + "⬜" * (10 - filled)

    return f"""
<div style="text-align:center; padding:18px; border-radius:16px; background:#fff8e8; border:2px solid #e7c87d;">
  <div style="font-size:84px; line-height:1;">{current['emoji']}</div>
  <h2 style="margin:8px 0;">{current['name']}</h2>
  <h3>⭐ {points} Plant Points</h3>
  <p><b>{progress_bar} {progress_pct}%</b></p>
  <p>Next: {next_text}</p>
  <p>{remaining_text}</p>
  <p>Completed actions: {int(profile.get('actions_count', 0) or 0)}</p>
</div>
"""


def get_gamification_outputs(user_id=DEFAULT_GAME_USER):
    try:
        profile = load_game_profile(user_id)
        events = load_game_events(user_id)
        status = "✅ Progress loaded from Firebase"
        return render_virtual_plant(profile), events, status
    except Exception as error:
        return "⚠️ Could not load the virtual plant.", pd.DataFrame(), f"❌ {error}"


def demo_add_care_action(user_id=DEFAULT_GAME_USER):
    """Visible demo action: adds 25 points so the lecturer can observe progress."""
    try:
        profile = award_game_points(
            action="demo_care_action",
            points=25,
            user_id=user_id,
            details="Manual demonstration from the Gamification tab",
        )
        events = load_game_events(user_id)
        return render_virtual_plant(profile), events, "✅ Demo action completed: +25 points"
    except Exception as error:
        return "⚠️ Could not update the plant.", pd.DataFrame(), f"❌ {error}"


def demo_advance_stage(user_id=DEFAULT_GAME_USER):
    """Adds exactly enough points to visibly move to the next plant stage."""
    try:
        profile = load_game_profile(user_id)
        _, next_stage = _stage_for_points(profile["points"])

        if next_stage is None:
            events = load_game_events(user_id)
            return render_virtual_plant(profile), events, "🌼 Maximum stage already reached"

        required = max(next_stage["min_points"] - profile["points"], 1)
        updated = award_game_points(
            action="demo_stage_advance",
            points=required,
            user_id=user_id,
            details=f"Demo advancement to {next_stage['name']}",
        )
        events = load_game_events(user_id)
        return (
            render_virtual_plant(updated),
            events,
            f"🎉 Stage advanced to {next_stage['emoji']} {next_stage['name']} (+{required} points)",
        )
    except Exception as error:
        return "⚠️ Could not advance the plant.", pd.DataFrame(), f"❌ {error}"


def reset_gamification(user_id=DEFAULT_GAME_USER):
    """Resets only the selected demo profile and removes its event history."""
    user_id = _safe_game_user(user_id)
    try:
        firebase_repo.db.collection("gamification_profiles").document(user_id).set({
            **_default_game_profile(user_id),
            "reset_at": firestore.SERVER_TIMESTAMP,
            "updated_at": firestore.SERVER_TIMESTAMP,
        })

        batch = firebase_repo.db.batch()
        deleted = 0
        for snapshot in firebase_repo.db.collection("gamification_events").stream():
            if snapshot.to_dict().get("user_id") == user_id:
                batch.delete(snapshot.reference)
                deleted += 1
                if deleted % 400 == 0:
                    batch.commit()
                    batch = firebase_repo.db.batch()
        batch.commit()

        profile = load_game_profile(user_id)
        return render_virtual_plant(profile), pd.DataFrame(), "🔄 Demo progress reset"
    except Exception as error:
        return "⚠️ Could not reset the plant.", pd.DataFrame(), f"❌ {error}"


# Wrappers preserve the existing UI outputs while awarding persistent points.
def gamified_analyze_plant_image(pil_image):
    result = analyze_plant_image(pil_image)
    if IMAGE_VALIDATION_PASSED:
        try:
            award_game_points("image_analysis", user_id=DEFAULT_GAME_USER)
        except Exception as error:
            print(f"Gamification warning: {error}")
    return result


def gamified_refresh_sensor_data(feed, limit):
    result = refresh_sensor_data(feed, limit)
    try:
        award_game_points("iot_refresh", user_id=DEFAULT_GAME_USER)
    except Exception as error:
        print(f"Gamification warning: {error}")
    return result


def gamified_run_gradio_search(question, search_type):
    # The interface uses a fixed value of 3 results and hides advanced details.
    result = run_gradio_search(question, search_type, n_results=3, show_advanced=False)
    if question and str(question).strip():
        try:
            award_game_points("rag_search", user_id=DEFAULT_GAME_USER)
        except Exception as error:
            print(f"Gamification warning: {error}")
    return result


def gamified_set_watering_reminder(plant_name, every_days, reminder_time):
    result = set_watering_reminder(plant_name, every_days, reminder_time)
    try:
        award_game_points("watering_reminder", user_id=DEFAULT_GAME_USER)
    except Exception as error:
        print(f"Gamification warning: {error}")
    return result


def gamified_log_health_entry(plant_name):
    result = log_health_entry(plant_name)
    try:
        award_game_points("health_checkin", user_id=DEFAULT_GAME_USER)
    except Exception as error:
        print(f"Gamification warning: {error}")
    return result


def gamified_chatbot_respond(message, history, session_id):
    result = chatbot_respond(message, history, session_id)
    if message and str(message).strip():
        try:
            award_game_points("chat_message", user_id=DEFAULT_GAME_USER)
        except Exception as error:
            print(f"Gamification warning: {error}")
    return result



In [ ]:
# Launch the Gradio application
if GRADIO_AVAILABLE:
    CUSTOM_CSS = f"""
.gradio-container {{
    background: {COLORS['bg']} !important;
    color: {COLORS['text']} !important;
}}

.gradio-container .prose,
.gradio-container p,
.gradio-container h1,
.gradio-container h2,
.gradio-container h3,
.gradio-container h4,
.gradio-container label,
.gradio-container span {{
    color: {COLORS['text']} !important;
}}

.gradio-container .block {{
    background: {COLORS['light']} !important;
    border: 1px solid #e7c87d !important;
    border-radius: 12px !important;
}}

.gradio-container button {{
    background: {COLORS['primary']} !important;
    color: white !important;
    border: none !important;
    border-radius: 8px !important;
}}

.gradio-container button:hover {{
    filter: brightness(0.95);
}}

.gradio-container input,
.gradio-container textarea {{
    background: #fff9ec !important;
    color: {COLORS['text']} !important;
    border: 1px solid #d9b96e !important;
}}

.gradio-container table {{
    background: white !important;
}}

.gradio-container .tab-nav button.selected {{
    color: {COLORS['primary']} !important;
    border-color: {COLORS['primary']} !important;
}}
"""
    with gr.Blocks(
    theme=gr.themes.Soft(),
    css=CUSTOM_CSS,
    title="OrangePanther — Calendula Cloud System"
) as interface:
        gr.Markdown("# 🌼 OrangePanther — Calendula Cloud System")
        gr.Markdown("Inverted Index + RAG + IoT dashboard + Plant Health History + AI Chatbot + Virtual Plant Gamification.")

        with gr.Tab("1. Image Upload"):
            gr.Markdown("Upload an image. Gemini first verifies that it contains a plant or flower; only then the RGB health analysis runs.")
            image_input = gr.Image(type="pil", label="Upload Plant Photo")
            image_btn = gr.Button("Analyze Image")
            image_plot = gr.Plot(label="Image Analysis")
            image_summary = gr.Markdown()
            image_btn.click(gamified_analyze_plant_image, inputs=image_input, outputs=[image_plot, image_summary])

        with gr.Tab("2. IoT Sensors"):
            gr.Markdown(f"Cloud server: `{BASE_URL}`. If the server is unavailable, the system uses simulated fallback data.")
            sensor_feed = gr.Dropdown(
                choices=["humidity", "soil", "temperature", "json"],
                value="json",
                label="Feed"
            )
            sensor_limit = gr.Slider(minimum=5, maximum=100, value=30, step=5, label="Number of samples")
            sensor_btn = gr.Button("Refresh Sensor Data")
            sensor_message = gr.Markdown()
            sensor_table = gr.Dataframe(label="Latest Sensor Rows")
            sensor_plot = gr.Plot(label="Sensor Trends")
            sensor_btn.click(gamified_refresh_sensor_data, inputs=[sensor_feed, sensor_limit], outputs=[sensor_message, sensor_table, sensor_plot])

        with gr.Tab("3. Search"):
            gr.Markdown("Ask about Calendula officinalis using semantic RAG or exact inverted-index search.")
            question = gr.Textbox(label="Question", placeholder="e.g. What are the medicinal properties of Calendula?", lines=2)
            search_type = gr.Radio(choices=["RAG (semantic)", "Inverted Index (exact)"], value="RAG (semantic)", label="Search mode")
            search_btn = gr.Button("Search")
            search_output = gr.Textbox(label="Results", lines=20)
            search_btn.click(
                gamified_run_gradio_search,
                inputs=[question, search_type],
                outputs=search_output
            )

        with gr.Tab("4. Dashboard"):
            gr.Markdown("Manager view: current status, latest sensor values, and care recommendations.")
            dash_plant = gr.Textbox(value="My Plant", label="Plant name")
            dash_btn = gr.Button("Update Dashboard")
            dash_summary = gr.Markdown()
            dash_plot = gr.Plot(label="Recent Sensor Trends")
            dash_btn.click(dashboard_status, inputs=dash_plant, outputs=[dash_summary, dash_plot])

            gr.Markdown("## Watering Reminders")
            gr.Markdown("Demo reminder setting for regular watering. This does not send a real push notification yet.")
            reminder_days = gr.Slider(minimum=1, maximum=14, value=3, step=1, label="Water every X days")
            reminder_time = gr.Textbox(value="09:00", label="Reminder time")
            reminder_btn = gr.Button("Set Watering Reminder")
            reminder_msg = gr.Markdown()
            reminder_btn.click(gamified_set_watering_reminder, inputs=[dash_plant, reminder_days, reminder_time], outputs=reminder_msg)

        with gr.Tab("5. Plant Health History"):
            gr.Markdown("Bonus feature: logs plant status over time and shows trends for the manager.")
            history_plant = gr.Textbox(value="Calendula", label="Plant name")
            with gr.Row():
                log_btn = gr.Button("Log Check-in")
                demo_btn = gr.Button("Load Demo")
                show_btn = gr.Button("Show History")
            history_message = gr.Markdown()
            history_table = gr.Dataframe(label="Last Check-ins")
            history_plot = gr.Plot(label="Health History")
            history_summary = gr.Markdown()
            log_btn.click(gamified_log_health_entry, inputs=history_plant, outputs=[history_message, history_table, history_plot, history_summary])
            demo_btn.click(seed_demo_history, inputs=history_plant, outputs=[history_message, history_table, history_plot, history_summary])
            show_btn.click(get_health_history_outputs, inputs=history_plant, outputs=[history_table, history_plot, history_summary])


        with gr.Tab("6. AI Chatbot"):
            gr.Markdown(
                "Chat with an intelligent assistant that uses Gemini, recent conversation context, "
                "the academic RAG sources, current IoT values, and Firebase chat history."
            )
            chat_session_id = gr.Textbox(
                value=DEFAULT_CHAT_SESSION,
                label="Chat session name",
                info="Use the same name later to reload this conversation."
            )
            chatbot = gr.Chatbot(
                label="OrangePanther AI Assistant",
                type="messages",
                height=430
            )
            chat_message = gr.Textbox(
                label="Your message",
                placeholder="Ask about Calendula, plant care, current sensor readings, or the application...",
                lines=2
            )
            with gr.Row():
                chat_send_btn = gr.Button("Send")
                chat_load_btn = gr.Button("Load Saved Chat")
                chat_clear_btn = gr.Button("Clear Chat")
            chat_status = gr.Markdown()

            chat_send_btn.click(
                gamified_chatbot_respond,
                inputs=[chat_message, chatbot, chat_session_id],
                outputs=[chatbot, chat_message, chat_status]
            )
            chat_message.submit(
                gamified_chatbot_respond,
                inputs=[chat_message, chatbot, chat_session_id],
                outputs=[chatbot, chat_message, chat_status]
            )
            chat_load_btn.click(
                load_chat_session,
                inputs=chat_session_id,
                outputs=[chatbot, chat_status]
            )
            chat_clear_btn.click(
                clear_chat_session,
                inputs=chat_session_id,
                outputs=[chatbot, chat_status]
            )


        with gr.Tab("7. Virtual Plant"):
            gr.Markdown(
                "## 🌱 Virtual Plant Gamification\n"
                "Complete actions across the application to collect Plant Points. "
                "The virtual plant grows through five stages, and all progress is stored in Firebase."
            )
            game_user_id = gr.Textbox(
                value=DEFAULT_GAME_USER,
                label="Gamification profile",
                info="Use demo_user to see points collected automatically from the other tabs."
            )
            game_card = gr.HTML(value=render_virtual_plant(_default_game_profile(DEFAULT_GAME_USER)))
            with gr.Row():
                game_load_btn = gr.Button("Load Progress")
                game_demo_btn = gr.Button("Add Demo Care Action (+25)")
                game_stage_btn = gr.Button("Advance to Next Stage")
                game_reset_btn = gr.Button("Reset Demo")
            game_status = gr.Markdown()
            game_events = gr.Dataframe(
                label="Recent Point Events — proof that the feature works",
                interactive=False
            )

            game_load_btn.click(
                get_gamification_outputs,
                inputs=game_user_id,
                outputs=[game_card, game_events, game_status]
            )
            game_demo_btn.click(
                demo_add_care_action,
                inputs=game_user_id,
                outputs=[game_card, game_events, game_status]
            )
            game_stage_btn.click(
                demo_advance_stage,
                inputs=game_user_id,
                outputs=[game_card, game_events, game_status]
            )
            game_reset_btn.click(
                reset_gamification,
                inputs=game_user_id,
                outputs=[game_card, game_events, game_status]
            )

    print("\n🚀 Launching Gradio interface...")
    interface.launch(share=True)
else:
    print("\n⚠️ Gradio not available — install gradio or use the classes directly.")